# Neighborhood Feature Extraction for Tumor Architecture
 
 **Author: Fabien Bertillot**
 
 This notebook extracts manuscript-aligned neighborhood features from a cell or ROI table with centroid coordinates and orientation vectors. It can be run independently when a compatible cell-feature table is available.
 
 Mask-level morphology features are handled separately in `morphological_feature_extraction.ipynb`.
 
 Feature families covered here:
 
 - **3-nearest-neighbor distance:** `Mean_Distance_K=3`, `Std_Distance_K=3`, `Cov_Distance_K=3`, and `Max_Distance_K=3`.
 - **Closest neighbor:** `Closest_Neighbor_Label` and `Closest_Distance`.
 - **Nuclear orientation order:** `Order_K=3`, using the local 2D nematic order parameter from neighboring cell orientation vectors.
 - **Stroma proximity:** optional tumor-cell centroid distance to the nearest stroma-cell centroid (`Dist_To_Stroma`).
 
 The coefficient of variation is reported as `Cov_Distance_K={k}` for consistency with the feature tables, although it represents std / mean rather than covariance.
 
 ---

## 1) Load Dependencies

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from neighborhood_utils import (
    add_k_neighbor_features,
    add_tumor_stroma_distance_features,
    summarize_neighborhood_features,
)

## 2) Configure Inputs
 
 `CELL_FEATURE_TABLE` should point to a table containing at least centroid coordinates and orientation. The utilities support common column names such as `Centroid_X`, `Centroid_Y`, `Orientation_X`, `Orientation_Y`, `Tissue_Type`, `x_centroid`, `y_centroid`, `x_orientation`, and `y_orientation`.
 
 Use `GROUP_COLS` to keep neighbors within each image/core/patient. This avoids accidentally linking cells across separate tissue sections.

In [ ]:
CELL_FEATURE_TABLE = Path(r"")
OUT_DIR = Path(r"")

X_COL = None  # Auto-detects x_centroid, Centroid_X, or X
Y_COL = None  # Auto-detects y_centroid, Centroid_Y, or Y
ANGLE_COL = None
X_ORIENTATION_COL = None  # Auto-detects x_orientation, Orientation_X, or orientation_x
Y_ORIENTATION_COL = None  # Auto-detects y_orientation, Orientation_Y, or orientation_y

K_VALUES = (3,)
GROUP_COLS = None  # Example: ["image_name"] or ["patient_id", "roi_id"]

TISSUE_COL = "Tissue_Type"
ADD_TUMOR_STROMA_DISTANCE = True
PIXEL_SIZE = 1.0

In [ ]:
# papermill passes parameters as strings
CELL_FEATURE_TABLE = Path(CELL_FEATURE_TABLE)
OUT_DIR = Path(OUT_DIR)

OUT_DIR.mkdir(parents=True, exist_ok=True)

## 3) Load Cell or ROI Feature Table

In [ ]:
if CELL_FEATURE_TABLE.suffix.lower() in {".xlsx", ".xls"}:
    df_cells = pd.read_excel(CELL_FEATURE_TABLE)
elif CELL_FEATURE_TABLE.suffix.lower() in {".h5", ".hdf", ".hdf5"}:
    df_cells = pd.read_hdf(CELL_FEATURE_TABLE)
else:
    df_cells = pd.read_csv(CELL_FEATURE_TABLE)

print(df_cells.shape)
df_cells.head()

## 4) Compute k-Neighbor Distance and Order Features

In [ ]:
df_neighbors = add_k_neighbor_features(
    df_cells,
    x_col=X_COL,
    y_col=Y_COL,
    k_values=K_VALUES,
    angle_col=ANGLE_COL,
    x_orientation_col=X_ORIENTATION_COL,
    y_orientation_col=Y_ORIENTATION_COL,
    group_cols=GROUP_COLS,
)

df_neighbors.head()

## 5) Optional: Add Tumor-to-Stroma Centroid Distance

When a `Tissue_Type` column is available, this computes the manuscript stroma-proximity feature as each tumor cell centroid's minimum Euclidean distance to stroma cell centroids.

In [ ]:
if ADD_TUMOR_STROMA_DISTANCE and TISSUE_COL in df_neighbors.columns:
    df_neighbors = add_tumor_stroma_distance_features(
        df_neighbors,
        tissue_col=TISSUE_COL,
        x_col=X_COL,
        y_col=Y_COL,
        tumor_feature_name="Dist_To_Stroma",
        stroma_feature_name="Dist_To_Tumor",
    )

df_neighbors.head()

## 6) Sanity-Check Feature Distributions

In [ ]:
plot_columns = [
    "Mean_Distance_K=3",
    "Std_Distance_K=3",
    "Cov_Distance_K=3",
    "Max_Distance_K=3",
    "Order_K=3",
    "Closest_Distance",
    "Dist_To_Stroma",
]
plot_columns = [col for col in plot_columns if col in df_neighbors.columns]

df_neighbors[plot_columns].hist(figsize=(14, 10), bins=30)
plt.tight_layout()

## 7) Summarize Per Image/Core

In [ ]:
df_neighbor_summary = summarize_neighborhood_features(
    df_neighbors,
    group_cols=GROUP_COLS,
)

df_neighbor_summary.head()

## 8) Save Feature Tables

In [ ]:
cell_out = OUT_DIR / "architecture_neighborhood_features_per_cell.csv"
summary_out = OUT_DIR / "architecture_neighborhood_features_summary.csv"

df_neighbors.to_csv(cell_out, index=False)
df_neighbor_summary.to_csv(summary_out, index=False)

print(f"Saved per-cell/ROI features to {cell_out}")
print(f"Saved summary features to {summary_out}")